# AAND Stage 1: Anomaly Amplification

In this notebook, we explore the first stage of the AAND (Anomaly Amplification and Normality Distillation) framework. The goal of this stage is to enhance the frozen DINOv2 teacher's ability to distinguish between normal and anomalous features.

We will look at:
1. **Anomaly Synthesis:** Generating fake defects using Perlin noise and procedural textures.
2. **Matching-guided Residual Gate (MRG):** Predicting where the anomalies are.
3. **Attribute-scaling Residual Generator (ARG):** Generating features that push anomalies away from normal representations.
4. **Feature Amplification:** How the final Advanced Teacher features are formed.

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as transforms
from glob import glob

from anomaly_synthesis import AnomalySynthesizer
from models import AdvancedDINOv2Teacher

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Anomaly Synthesis
Since we don't have enough real anomalous images during training, we synthesize fake ones. We use Perlin noise to create a mask, and blend procedural textures into a normal image.

In [ ]:
# Load a sample normal image
images = sorted(glob('dataset/train/images/*.*'))
if len(images) == 0:
    print("No images found. Creating a dummy image.")
    img = Image.fromarray(np.random.randint(50, 200, (518, 518, 3), dtype=np.uint8))
else:
    img = Image.open(images[0]).convert('RGB').resize((518, 518))

# Create synthesizer
synthesizer = AnomalySynthesizer(image_size=518, texture_source='procedural')

# Generate corrupted image
corrupted_img, mask = synthesizer(img)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img)
axes[0].set_title("Original (Normal) Image")
axes[0].axis('off')

axes[1].imshow(mask, cmap='gray')
axes[1].set_title("Synthetic Anomaly Mask (Perlin)")
axes[1].axis('off')

axes[2].imshow(corrupted_img)
axes[2].set_title("Corrupted Image")
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 2. Advanced Teacher & Residual Modules
Now we pass this corrupted image through the `AdvancedDINOv2Teacher`. 

The teacher has a standard frozen backbone, but at the last block, it uses the **Residual Anomaly Amplification (RAA)** module.

In [ ]:
# Initialize Advanced Teacher
teacher = AdvancedDINOv2Teacher(size='large').to(device)

# Transform input
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
input_tensor = transform(corrupted_img).unsqueeze(0).to(device)
gt_mask = torch.from_numpy(mask).unsqueeze(0).unsqueeze(0).float() # (1, 1, 518, 518)
gt_mask_downsampled = torch.nn.functional.interpolate(gt_mask, size=(37, 37), mode='nearest').to(device)

# Forward pass
# Note: during training we pass gt_mask so the residual generator uses ground truth
adv_features, pred_weights, vanilla_features = teacher(input_tensor, gt_mask=gt_mask_downsampled)

print("Vanilla Teacher Features:", vanilla_features.shape)
print("Advanced Features:", adv_features.shape)
print("Predicted Anomaly Weights:", pred_weights.shape)

### Visualizing the MRG Prediction (Gate)
The Matching-guided Residual Gate (MRG) predicts which patches are anomalous. It learns to do this via Focal Loss against the synthetic mask.

In [ ]:
import torch.nn.functional as F

# Upsample the predicted weights for visualization
pred_weights_vis = F.interpolate(pred_weights, size=(518, 518), mode='bilinear').squeeze().cpu().detach().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(corrupted_img)
axes[0].set_title("Input Image")
axes[0].axis('off')

axes[1].imshow(mask, cmap='gray')
axes[1].set_title("Ground Truth Mask")
axes[1].axis('off')

im = axes[2].imshow(pred_weights_vis, cmap='jet', vmin=0, vmax=1)
axes[2].set_title("MRG Predicted Weights (Gate)")
axes[2].axis('off')
plt.colorbar(im, ax=axes[2])

plt.tight_layout()
plt.show()

### 3. Amplifying Anomalies
The Attribute-scaling Residual Generator (ARG) creates residuals for every patch. The MRG gate then decides how much of this residual to add to the Vanilla features.

`Advanced_Features = Vanilla_Features + MRG_Gate * ARG_Residual`

Let's look at the difference between the Vanilla features and the Advanced features (i.e. the gated residual).

In [ ]:
# Calculate the gated residual (Difference between Advanced and Vanilla)
gated_residual = (adv_features - vanilla_features).abs().mean(dim=1, keepdim=True)
gated_residual_vis = F.interpolate(gated_residual, size=(518, 518), mode='bilinear').squeeze().cpu().detach().numpy()

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(corrupted_img)
ax[0].imshow(mask, cmap='jet', alpha=0.3)
ax[0].set_title("Input with Anomaly Overlay")
ax[0].axis('off')

im = ax[1].imshow(gated_residual_vis, cmap='hot')
ax[1].set_title("Gated Residual Magnitude (Feature Diff)")
ax[1].axis('off')
plt.colorbar(im, ax=ax[1])

plt.show()

As we can see, the residual is **suppressed on normal patches** and **amplified on anomalous patches**. 

This means the pre-trained normal knowledge is protected, while the anomalies are pushed further out of distribution, fulfilling the core assumption of the Teacher-Student architecture!